# NCCL 백엔드를 모사하여 단순 파이썬으로 Ring All-Reduce 로직 구현

> 본 노트북은 SYLLABUS.md에 기반하여 자동 생성된 뼈대(Skeleton) 파일입니다. 상세한 이론, 수식 및 코드는 추가로 구현되어야 합니다.

## 1. 개요 및 학습 목표
이 노트북에서는 해당 주제에 대한 핵심 개념을 다룹니다.

## 2. 핵심 이론 및 수학적 원리
- 수식 및 상세한 동작 원리를 여기에 기록합니다.

## 3. 실습 코드 구현
아래 셀을 통해 파이썬 및 관련 프레임워크 코드를 직접 작성하고 실행해 볼 수 있습니다.

In [ ]:
# 실습을 위한 기본 라이브러리 임포트
import tensorflow as tf
import numpy as np

print(f"TensorFlow Version: {tf.__version__}")


---
## Q1: Ring Topology 정의

분산 처리에서 Ring All-Reduce는 N개의 워커들이 링(Ring) 형태로 연결되어 그래디언트를 교환합니다. 이 링 구조를 파이썬 딕셔너리로 표현하세요.

**요구사항:**
- `num_workers = 4`라고 할 때, 각 워커의 `왼쪽 이웃(prev)`과 `오른쪽 이웃(next)` 워커 인덱스를 담은 딕셔너리 `ring_topology`를 만드세요.
- 예: 워커 0의 next는 1, prev는 3 (원형 순환).
- 딕셔너리를 출력하여 링 구조를 확인하세요.

In [ ]:
# TODO: 코드를 직접 작성해 보세요.

<details>
<summary>💡 정답 보기</summary>

```python
import numpy as np

num_workers = 4

ring_topology = {
    i: {
        'prev': (i - 1) % num_workers,
        'next': (i + 1) % num_workers
    }
    for i in range(num_workers)
}

print("Ring Topology:")
for worker, neighbors in ring_topology.items():
    print(f"  Worker {worker}: prev={neighbors['prev']}, next={neighbors['next']}")
```
</details>

---
## Q2: Scatter-Reduce 단계 시뮬레이션

Ring All-Reduce의 1단계인 Scatter-Reduce를 시뮬레이션하세요. 각 워커가 자신의 그래디언트 청크를 오른쪽 이웃에게 보내고, 왼쪽에서 받은 값과 자신의 값을 합산합니다.

**요구사항:**
- 각 워커가 크기 8인 랜덤 그래디언트 배열을 가진다고 가정하세요: `grads = [np.random.rand(8) for _ in range(num_workers)]`
- `num_workers - 1`번 반복하며 각 스텝에서 인접 워커 간 청크 합산을 시뮬레이션하세요.
- 최종적으로 각 워커가 담당하는 청크에는 모든 워커의 합이 들어 있어야 합니다.

In [ ]:
# TODO: 코드를 직접 작성해 보세요.

<details>
<summary>💡 정답 보기</summary>

```python
import numpy as np

num_workers = 4
chunk_size = 2  # 8개 그래디언트를 4조각으로

np.random.seed(42)
grads = [np.random.rand(8) for _ in range(num_workers)]
true_sum = sum(grads)  # 정답: 실제 합산 값

# 각 워커별 청크 보관 (워커 i는 chunk_size * i ~ chunk_size * (i+1) 담당)
buffers = [g.copy() for g in grads]

for step in range(num_workers - 1):
    new_buffers = [b.copy() for b in buffers]
    for i in range(num_workers):
        send_chunk_idx = (i - step) % num_workers
        recv_from = (i - 1) % num_workers
        recv_chunk_idx = (recv_from - step) % num_workers

        s = recv_chunk_idx * chunk_size
        e = s + chunk_size
        new_buffers[i][s:e] += buffers[recv_from][s:e] - grads[recv_from][s:e]
    buffers = new_buffers

print("Scatter-Reduce 검증 (워커 0 첫 청크):")
print(f"  계산값: {buffers[0][:chunk_size]}")
print(f"  참값: {true_sum[:chunk_size]}")
```
</details>

---
## Q3: All-Gather 단계 시뮬레이션

Ring All-Reduce의 2단계인 All-Gather를 시뮬레이션하세요. Scatter-Reduce 결과로 각 워커가 하나의 청크에만 최종 합산값이 있는 상태에서, 이를 모든 워커에게 전파하세요.

**요구사항:**
- Q2의 Scatter-Reduce 결과를 이어받아 `num_workers - 1`번 반복하며 최종 합산 청크를 링을 따라 전파하세요.
- 모든 반복이 끝나면 모든 워커의 배열이 `true_sum`과 일치해야 합니다.
- 최대 절대 오차(max absolute error)를 계산하여 정확성을 검증하세요.

In [ ]:
# TODO: 코드를 직접 작성해 보세요.

<details>
<summary>💡 정답 보기</summary>

```python
# All-Gather: 각 워커가 가진 완성된 청크를 오른쪽으로 전파
for step in range(num_workers - 1):
    new_buffers = [b.copy() for b in buffers]
    for i in range(num_workers):
        recv_from = (i - 1) % num_workers
        propagate_chunk_idx = (i - step) % num_workers
        s = propagate_chunk_idx * chunk_size
        e = s + chunk_size
        new_buffers[i][s:e] = buffers[recv_from][s:e]
    buffers = new_buffers

# 검증
errors = [np.max(np.abs(buffers[i] - true_sum)) for i in range(num_workers)]
print("All-Gather 검증:")
for i, err in enumerate(errors):
    print(f"  워커 {i} 최대 오차: {err:.2e}")
print(f"\n모든 워커 동기화 성공: {all(e < 1e-10 for e in errors)}")
```
</details>